In [ ]:
import _plotting as plot
import jax
import jax.numpy as jnp
from matplotlib import pyplot as plt

from xxm.core.align import match_states_by_mean as match_states
from xxm.hmm import GaussianHMM

In [ ]:
def make_true_model() -> GaussianHMM:
    return GaussianHMM.from_params(
        initial_probs=jnp.array([0.4, 0.3, 0.3]),
        transition_probs=jnp.array(
            [
                [0.90, 0.08, 0.02],
                [0.05, 0.90, 0.05],
                [0.03, 0.07, 0.90],
            ]
        ),
        emission_means=jnp.array(
            [
                [-3.0, -2.0],
                [0.0, 3.0],
                [3.0, -1.0],
            ]
        ),
        emission_covariances=jnp.array(
            [
                [[0.6, 0.2], [0.2, 0.5]],
                [[0.8, -0.2], [-0.2, 0.5]],
                [[0.5, 0.1], [0.1, 0.8]],
            ]
        ),
    )


true_model = make_true_model()

true_states, observations = true_model.sample(
    num_steps=1000,
    key=jax.random.key(0),
)

In [ ]:
_, ax = plt.subplots()
plot.plot_seq_2d(ax, true_states, observations)

In [ ]:
initial_model = GaussianHMM.from_kmeans(
    observations=observations,
    num_states=true_model.num_states,
    key=jax.random.key(0),
)

fit = initial_model.fit(
    observations=observations,
    num_iters=50,
)


plot.plot_fit_progress(fit.objective_trace)

In [ ]:
def align_model(learned_model, true_model):

    permutation = match_states(
        learned_model.states.mean,
        true_model.states.mean,
    )
    learned_model = learned_model.permute(permutation)

    return learned_model


learned_model = align_model(fit.model, true_model)

In [ ]:
def plot_emissions(
    observations: jnp.ndarray,
    true_model: GaussianHMM,
    learned_model: GaussianHMM,
) -> None:
    _f, ax = plt.subplots(figsize=(6, 6))

    ax.scatter(
        observations[:, 0],
        observations[:, 1],
        s=5,
        alpha=0.1,
    )

    for i in range(true_model.num_states):
        plot.plot_gaussian_2d_ellipse(
            ax,
            true_model.states.select(i),
            linestyle='--',
            label='True' if i == 0 else None,
            color=f'C{i}',
        )

    for i in range(learned_model.num_states):
        plot.plot_gaussian_2d_ellipse(
            ax,
            learned_model.states.select(i),
            linestyle='-',
            label='Learned' if i == 0 else None,
            color=f'C{i}',
        )

    ax.set_xlabel('Observation 1')
    ax.set_ylabel('Observation 2')
    ax.set_aspect('equal')
    ax.legend()


plot_emissions(
    observations,
    true_model,
    learned_model,
)

In [ ]:
posterior, _ = learned_model.infer(observations)

In [ ]:
plot.plot_seq_1d_comparison(
    true_states,
    observations,
    learned_model.most_likely_states(posterior),
    learned_model.observation_mean(posterior),
)